In [534]:
import RandomSquareGenerator
import numpy as np
import matplotlib.pyplot as plt
from statistics import mean
from collections import Counter, defaultdict
import hashlib
from math import comb
from numpy.random import permutation
from LatinSquares import reduce_square

### Step 1: Helper functions

In [535]:
def canonicalize(square):
    '''
    Converts a square (list of lists, NumPy array, etc.) into an immutable
    tuple-of-tuples form so it can be hashed or used as a dictionary key.
    '''
    return tuple(map(tuple, square))

def random_isotopy(square):
    '''
    Apply a random isotopy to a Latin square:
      1. Randomly permute rows.
      2. Randomly permute columns.
      3. Randomly permute symbols.

    Returns the transformed square in canonicalized form.
    '''
    square = np.array(square)
    n = square.shape[0]
    
    # Random row permutation
    row_perm = permutation(n)
    square = square[row_perm, :]

    # Random column permutation
    col_perm = permutation(n)
    square = square[:, col_perm]

    # Random symbol permutation
    symbol_perm = permutation(n) 
    symbol_map = {i: symbol_perm[i] for i in range(n)}
    square = np.vectorize(lambda x: symbol_map[x])(square)

    return canonicalize(square)

### Step 2: Sampling function

In [540]:
def run_sampling_trials(square, num_trials, sample_size):
    '''
    Run trials and collect collision counts per trial.
    Returns: dict mapping multiplicity k -> list of counts across trials
    '''
    # e.g., {2: [14, 12, 15, ...], 3: [1, 0, 2, ...], ...}
    # where 2: [14, 12, ...] means that trial one had 14 pairs, trial two had 12..
    # default value for any new entry is an empty list 
    # ? clarify trials used to get isotopy class estimate.
    # ? may not need this distribution dictionary
    collision_distributions = defaultdict(list)

    for trial in range(num_trials):
        # store sampled latin sqaures
        sample = []

        for _ in range(sample_size):
            # Get random square in isotopy class
            isotopy_square = random_isotopy(square.store)
            
            # Transform square to common form and canoncalize
            square_reduced = reduce_square(isotopy_square)
            square_flat_reduced = canonicalize(square_reduced)
            
            # add to sample
            sample.append(square_flat_reduced)
        
        # Counts occurances of each square
        collision_count = Counter(sample)
        # Counts multiplicities ex. 14 singles, 5 doubles, 1 triple 
        multiplicities_count = Counter(collision_count.values())

        # Number of multiplicity levels to track
        for k in range(1, 5):
            collision_distributions[k].append(multiplicities_count.get(k, 0))

    return collision_distributions

### Step 2. Estimate Population Size $\hat{N}$

In [537]:
def estimate_population_size(collision_counts:defaultdict, sample_size):
    '''
    Estimate the total population size from observed collision counts.
    
    TODO (LOOK INTO THIS) Triple collisions if the average is > 10 (formula based on C(n, 3)).
    Otherwise, pair collisions (formula based on C(n, 2)). Returns infinity if no collisions are observed.
    '''
    avg_pairs = mean(collision_counts[2]) if collision_counts[2] else 0 
    avg_triples = mean(collision_counts[3]) if collision_counts[3] else 0
    
    if avg_triples > 10:
        estimated_N = (comb(sample_size, 3) / avg_triples) ** (1 / 2)
    elif avg_pairs > 0:
        estimated_N = comb(sample_size,2) / avg_pairs
    else:
        estimated_N = float('inf')
    return estimated_N

### Step 3: Combine Previous Steps


In [538]:
def estimate_neighbours_class_sizes(neighbours,sample_size):
    # Generate Square and make duplicate to shuffle 
    # ! or load square of interest into rlatin class 
    square_0 = RandomSquareGenerator.rlatin(5,Minmoves=25)
    square_0_duplicate = RandomSquareGenerator.rlatin(len(square_0.store[0]), source=square_0, Minmoves=25)
        
    # Stores square, estimated isotopy class size, and collison
    results = []
    
    for _ in range(neighbours):
        square_0_duplicate.shuffle()
        
        square_1 = square_0_duplicate
        
        # Estimate class size
        # ? will need to update sample_size
        collision_distribution = run_sampling_trials(square_1, 20, sample_size)   
        estimated_N = estimate_population_size(collision_distribution,sample_size)
        
        results.append({
            "square": square_1.store,
            "estimated_N": estimated_N, 
            "collisions": collision_distribution
        })

    return results

In [ ]:
result = estimate_neighbours_class_sizes(neighbours=100,sample_size=30)

In [542]:
result

[{'square': array([[3, 1, 4, 0, 2],
         [4, 3, 0, 2, 1],
         [0, 2, 1, 4, 3],
         [1, 0, 2, 3, 4],
         [2, 4, 3, 1, 0]], dtype=int16),
  'estimated_N': 96.66666666666667,
  'collisions': defaultdict(list,
              {1: [19,
                14,
                15,
                18,
                21,
                16,
                17,
                20,
                16,
                19,
                17,
                18,
                15,
                18,
                22,
                16,
                17,
                22,
                17,
                15],
               2: [4, 8, 6, 6, 1, 4, 2, 5, 7, 4, 2, 6, 3, 4, 4, 4, 5, 4, 5, 6],
               3: [1, 0, 1, 0, 1, 2, 3, 0, 0, 1, 3, 0, 3, 0, 0, 2, 1, 0, 1, 1],
               4: [0,
                0,
                0,
                0,
                1,
                0,
                0,
                0,
                0,
                0,
                0,